# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")
df.display()

# Silver Transformation

Dynamically trims all string columns in a DataFrame, rather than manually specifying each column name.

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Customer ID Cleaning

In [0]:

df = df.withColumn(
    "cid",
    F.when(col("cid").startswith("NAS"),
           F.substring(col("cid"), 4, F.length(col("cid"))))
     .otherwise(col("cid"))
)
df.display()

# Birthdate Validation

In [0]:
# Sets 'bdate' to null if it is a future date; otherwise keeps the original value
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(), None)
     .otherwise(col("bdate"))
)

# Gender Normalization

In [0]:
df = df.withColumn(
    "gen",
    F.when(F.upper(col("gen")).isin("F","FEMALE"),
           "Female")
     .when(F.upper(col("gen")).isin("M","MALE"),
           "Male")
     .otherwise("n/a")
)
df.display()


# Rename Columns

In [0]:
rename_map={
    "cid":"customer_number",
    "bdate":"birth_date",
    "gen":"gender"
    }

for old_name,new_name in rename_map.items():
    df=df.withColumnRenamed(old_name,new_name)


In [0]:
df.limit(10).display()

# Write Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")

In [0]:
%sql
select * from workspace.silver.erp_customers limit 5